# Upwork API Technical Support Bot — RAG Pipeline
**Associate AI Developer Assignment**

This notebook walks through the full pipeline:
- **Part A**: Knowledge Engineering (Load → Chunk → Embed → Store)
- **Part B**: RAG Implementation (Retrieve → Prompt → Answer)
- **Part C**: Evaluation (3 ground-truth questions)

## 0. Setup — Install Dependencies

In [ ]:
# Run once to install all packages
# !pip install -r requirements.txt

In [ ]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv

# Load API key from .env
load_dotenv()
DEEPINFRA_API_KEY = os.getenv('DEEPINFRA_API_KEY')
assert DEEPINFRA_API_KEY, 'Set DEEPINFRA_API_KEY in your .env file!'
print('✅ API key loaded.')

## Part A — Knowledge Engineering

### A1. Load the PDF Documentation (Sanity Check)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = 'API_Documentation_Partial.pdf'  # <-- update path if needed

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

# --- Sanity Check ---
total_chars = sum(len(doc.page_content) for doc in documents)
print(f'Total pages loaded   : {len(documents)}')
print(f'Total character count: {total_chars}')
print(f'\nSample text (first 500 chars):')
print(documents[0].page_content[:500])

### A2. Document Chunking

**Why overlap matters:** Code snippets, endpoint URLs, and parameter descriptions often span sentence or line boundaries. A 50-character overlap ensures that context shared between adjacent chunks isn't lost — for example, an HTTP endpoint URL that starts at the end of one chunk will also appear at the beginning of the next, so retrieval always captures the complete unit of meaning.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

chunks = splitter.split_documents(documents)
print(f'Total chunks created: {len(chunks)}')
print(f'\nSample chunk:')
print(chunks[0].page_content)

### A3. Embed & Store in ChromaDB

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

DB_DIR = './chroma_db'

print('Loading local embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
)

print('Building ChromaDB vector store...')
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR,
)
print(f'✅ Vector store built and saved to {DB_DIR}')

## Part B — RAG Implementation

### B1. Semantic Retrieval

In [ ]:
def retrieve_chunks(query, vector_store, k=3):
    """Return top-k most relevant chunks for a given query."""
    retriever = vector_store.as_retriever(search_kwargs={'k': k})
    return retriever.invoke(query)

# Test it
test_query = 'How long is an OAuth access token valid for?'
results = retrieve_chunks(test_query, vector_store)
print(f'Top {len(results)} chunks for: "{test_query}"\n')
for i, doc in enumerate(results, 1):
    print(f'--- Chunk {i} (page {doc.metadata.get("page", "?")}) ---')
    print(doc.page_content[:300])
    print()

### B2. API Integration & Prompting

In [ ]:
import requests

DEEPINFRA_API_URL = 'https://api.deepinfra.com/v1/openai/chat/completions'
MODEL_NAME        = 'meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo'

SYSTEM_PROMPT = """You are a Senior Upwork API Consultant with 10+ years of experience.
You help developers integrate with the Upwork API accurately and efficiently.

STRICT RULES:
1. Answer ONLY using the provided context chunks.
2. If the answer is NOT in the context, say exactly:
   "I'm sorry, but the provided documentation does not contain that information."
3. Be precise and technical.
"""

def answer_question(user_query, retrieved_chunks):
    # Build context string
    context_text = ''
    for i, doc in enumerate(retrieved_chunks, 1):
        page = doc.metadata.get('page', '?')
        context_text += f'\n--- Source {i} (Page {page}) ---\n{doc.page_content}\n'

    user_message = f"""Use ONLY the following documentation excerpts to answer the question.

CONTEXT:
{context_text}

QUESTION: {user_query}

ANSWER:"""

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_message},
    ]

    headers = {
        'Content-Type':  'application/json',
        'Authorization': f'Bearer {DEEPINFRA_API_KEY}',
    }
    payload = {
        'model':       MODEL_NAME,
        'messages':    messages,
        'temperature': 0.1,
        'max_tokens':  512,
    }

    start = time.time()
    resp  = requests.post(DEEPINFRA_API_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    latency = round(time.time() - start, 2)

    answer = resp.json()['choices'][0]['message']['content'].strip()
    return {'answer': answer, 'latency': latency, 'sources': retrieved_chunks}

# Quick test
result = answer_question(test_query, results)
print(f'Answer  : {result["answer"]}')
print(f'Latency : {result["latency"]}s')

## Part C — Evaluation (Ground Truth Questions)

In [ ]:
eval_questions = [
    'What is the specific request-per-second rate limit for the Upwork API, and is it enforced per Key or per IP?',
    'How long is an OAuth access token valid for?',
    'Can I use a Client Credentials Grant to access a user\'s private contract details?',
]

for q in eval_questions:
    print(f'\n{'='*60}')
    print(f'Q: {q}')
    print('='*60)
    chunks  = retrieve_chunks(q, vector_store)
    result  = answer_question(q, chunks)
    print(f'A: {result["answer"]}')
    print(f'   Latency: {result["latency"]}s')